# 11 · M5 replay exam

One Walmart shop, `shop-CA_1`, with one `StaticFactoryPolicy` factory per item at lead time 3.
Five FOODS items are chosen on calendar 2015. The zero-shot row scores the
synthetic planner without fitting on either M5 window. The adaptation row
refits on reward-free transitions from 2015 only and is scored on
2016-01-01 through 2016-04-24. Both ordering policies sit inside
`PriceReplayPolicy` under `flat_world`. The textbook anchor is an
`OrderUpToPolicy` tuned on 2015 replay profit. Neither row is a pass/fail gate.

Raw files are git-ignored. When the raw directory is absent this notebook
skips and prints the Kaggle download plus the ADR 0028 slice rule.


In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "pyproject.toml").exists():
    os.chdir("..")

from src.jepa.m5 import skip_message

data_dir = Path(os.environ.get("JEPA_M5_DATA_DIR", "data/m5"))
message = skip_message(data_dir)
M5_AVAILABLE = message is None
if message:
    print(message)
else:
    print("M5 files found at", data_dir.resolve())


## Slice

ADR 0028. Rank `FOODS_3` at `CA_1` on 2015-01-01 through 2015-12-31 by shortest
`longest_zero_run`, then highest mean sales. Drop a zero run longer than 7 days.
If fewer than five items survive, widen to all FOODS departments at `CA_1`.
The five ids are written into this notebook only after that quality report runs.
Until then `PINNED_ITEM_IDS` stays empty, so a checkout without the Kaggle files
does not invent a slice.


In [ ]:
from src.datasets.m5 import load_m5_slice, quality_report
from src.jepa.m5 import (
    STORE_ID,
    load_selection_slice,
    pin_item_ids,
    rank_foods_items,
    resolve_pinned_items,
)

# Empty until the first quality report. pin_item_ids writes the five ids here.
PINNED_ITEM_IDS: list[str] = []

if not M5_AVAILABLE:
    print("[skipped — M5 data absent]")
else:
    selection = load_selection_slice(data_dir)
    report = quality_report(selection)
    ranked = rank_foods_items(selection)
    item_ids = resolve_pinned_items(PINNED_ITEM_IDS, ranked)
    if not PINNED_ITEM_IDS:
        pin_item_ids(Path("notebooks/11-jepa-m5.ipynb"), item_ids)
        PINNED_ITEM_IDS = list(item_ids)
    print(report[report["item_id"].isin(item_ids)])
    print("pinned", item_ids)


## Rows

The test window is 2016-01-01 through 2016-04-24. Zero-shot uses synthetic
weights and does not fit on either M5 window. Adaptation refits on the 2015
replay graph only, from `OrderUpToPolicy` and `ExploratoryPolicy`. The windows
are not pooled. The anchor search uses `order_up_to_space` on 2015 replay
profit. The synthetic log this cell fits is the current collector: the
recorded action is the filled quantity, in-transit is an arrival-age profile,
and the sales window is not the delivery lag. Imagined arrivals use the
textbook arms' fill rate from that short log. This exam is not the
holding-rate sweep. This walkthrough uses a short synthetic fit and two anchor
trials so it can finish; `replay_exam` defaults are the published search (150
trials, budget 64). Both rows are printed. Neither is a pass/fail gate.


In [ ]:
from src.jepa.log import collect_transition_log
from src.jepa.m5 import SELECTION_END, SELECTION_START, TEST_END, TEST_START, replay_exam, write_replay_exam
from src.jepa.plan import fit_latent

if not M5_AVAILABLE:
    print("[skipped — M5 data absent]")
elif not item_ids:
    print("No FOODS items survived the ADR 0028 slice rule.")
else:
    synthetic_log = collect_transition_log(
        n_episodes_each=1,
        episode_length=8,
        k_active=2,
        delivery_lag=3,
    )
    synthetic = fit_latent(
        synthetic_log,
        delivery_lag=3,
        epochs=1,
        budget=4,
        policy_seed=0,
    )
    adaptation = load_m5_slice(
        data_dir, item_ids, [STORE_ID], SELECTION_START, SELECTION_END,
    )
    held_out = load_m5_slice(
        data_dir, item_ids, [STORE_ID], TEST_START, TEST_END,
    )
    exam = replay_exam(adaptation, held_out, synthetic, n_trials=2, epochs=1, budget=4)
    out = write_replay_exam(exam, Path("runs/jepa-replenishment/m5"))
    for row in exam.rows:
        print(
            row.label,
            "planner", row.planner_net_profit,
            "anchor", row.anchor_net_profit,
        )
    print("2015 anchor profit", exam.anchor_selection_profit)
    print("neither row is a pass/fail gate")
    print("wrote", out)
